# 🍷 Wine Quality Analysis & Prediction Project

## Project Overview
This project aims to analyze the chemical properties of red wine and predict its quality. We will classify wines into two categories: **GOOD** (Quality >= 7) and **BAD** (Quality < 7).

### Key Steps:
1. Data Loading and Basic EDA
2. Missing Value Analysis & Correlation
3. Data Preprocessing & Binary Classification Setup
4. Logistic Regression (With and Without Scaling)
5. Model Comparison (Logistic Regression, KNN, Decision Tree)
6. Hyperparameter Tuning using GridSearchCV
7. Feature Importance Analysis

## 1. Import Libraries and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Set aesthetic style for plots
sns.set(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 8)

# Load dataset from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=';')

print("Dataset loaded successfully!")

## 2. Basic Data Analysis
Let's explore the dataset using `head()`, `info()`, and `describe()`.

In [ ]:
print("--- First 5 Rows ---")
display(df.head())

print("\n--- Dataset Info ---")
df.info()

print("\n--- Descriptive Statistics ---")
display(df.describe())

## 3. Missing Values and Correlation Analysis

In [ ]:
print("--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Correlation with Quality column ---")
correlations = df.corr()['quality'].sort_values(ascending=False)
print(correlations)

# Visualizing Correlations
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()

## 4. Target Conversion (Quality Labeling)
We convert the `quality` column into a binary target `quality_label`.
- Quality >= 7 -> GOOD (1)
- Quality < 7 -> BAD (0)

In [ ]:
df['quality_label'] = df['quality'].apply(lambda x: 1 if x >= 7 else 0)

print("Value Counts for quality_label:")
print(df['quality_label'].value_counts())

# Visualize the distribution
sns.countplot(x='quality_label', data=df)
plt.title('Distribution of Quality Label (0=Bad, 1=Good)')
plt.show()

## 5. Train-Test Split

In [ ]:
# Separate features and target
X = df.drop(['quality', 'quality_label'], axis=1)
y = df['quality_label']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 6. Logistic Regression (Without Scaling)

In [ ]:
lr_unscaled = LogisticRegression(max_iter=10000)
lr_unscaled.fit(X_train, y_train)
y_pred_unscaled = lr_unscaled.predict(X_test)

def evaluate_model(y_true, y_pred, title="Model Evaluation"):
    print(f"\n--- {title} ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall: {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_true, y_pred):.4f}")
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

evaluate_model(y_test, y_pred_unscaled, "Logistic Regression (No Scaling)")

## 7. Scaling and Retraining
We apply `StandardScaler` to see if it improves Logistic Regression performance.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_scaled = LogisticRegression()
lr_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = lr_scaled.predict(X_test_scaled)

evaluate_model(y_test, y_pred_scaled, "Logistic Regression (With Scaling)")

## 8. Model Comparison
We compare Logistic Regression, K-Nearest Neighbors (KNN), and Decision Tree.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

results = []

for name, model in models.items():
    # Use scaled data for LR and KNN, but Decision Tree doesn't strictly need it
    if name == "Decision Tree":
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    else:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
display(results_df.sort_values(by='F1-Score', ascending=False))

## 9. Hyperparameter Tuning (GridSearchCV)
We will tune the Decision Tree model as it often shows significant change with depth.

In [ ]:
param_grid = {
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='f1')
dt_grid.fit(X_train, y_train)

print(f"Best Parameters: {dt_grid.best_params_}")
best_model = dt_grid.best_estimator_

y_pred_tuned = best_model.predict(X_test)
evaluate_model(y_test, y_pred_tuned, "Tuned Decision Tree")

## 10. Feature Importance Analysis

In [ ]:
importances = best_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Feature Importance (Decision Tree)')
plt.show()

print("Top 3 important features:")
print(feature_importance_df.head(3))